> **Research record.** Cleaned copy of `XXX-calculate-WFH/100-remote-work-results-v2.2-results.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Diagnostics that justify keeping user-months with `total_work_days >= 7` before computing the remote-work metric.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

# Remote-work threshold diagnostics

This cleaned notebook keeps only the diagnostics used to justify filtering user-month records with `total_work_days >= 7`.

The input CSVs are produced by `003-remote-work-monthly.py` as `processed/dwells/remote_work/user_remote_work_{MONTH}.csv`.

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = "../.."
if project_root not in sys.path:
    sys.path.append(project_root)

from geofunctions import utils

MIN_TOTAL_WORK_DAYS = 7
REMOTE_WORK_FEATURE = "flag_remote_work_smoothed_max_thresh"
sns.set_theme(style="whitegrid")

In [ ]:
months = utils.return_list_months_ok()
list_months = []

for month in months:
    try:
        df_part = pd.read_csv(
            utils.get_path("processed", "dwells", f"remote_work/user_remote_work_{month}.csv")
        )
        df_part["month"] = month
        list_months.append(df_part)
    except FileNotFoundError:
        print(f"Missing user_remote_work file for {month}")

df_shares = pd.concat(list_months, ignore_index=True)
df_shares["undefined_days"] = (
    df_shares["total_work_days"]
    - df_shares["office_day_counts"]
    - df_shares["remote_work_days_min_work"]
)

## Distribution of detected work days

A user-month with very few detected work days gives an unstable WFH share. The red line marks the chosen threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df_shares["total_work_days"], bins=30, ax=ax)
ax.axvline(MIN_TOTAL_WORK_DAYS, color="red", linestyle="--", label=f"threshold = {MIN_TOTAL_WORK_DAYS}")
ax.set_title("Distribution of total detected work days per user-month")
ax.set_xlabel("total_work_days")
ax.set_ylabel("User-months")
ax.legend()
plt.show()

## Retention tradeoff by threshold

This plot shows how much sample is retained as the minimum `total_work_days` threshold changes. The selected value should remove low-information user-months while preserving enough observations.

In [ ]:
thresholds = list(range(1, 16))
retention = []

for threshold in thresholds:
    kept = df_shares[df_shares["total_work_days"] >= threshold]
    retention.append(
        {
            "threshold": threshold,
            "retained_user_month_share": len(kept) / len(df_shares),
            "retained_unique_users": kept["identifier"].nunique(),
            "mean_wfh": kept[REMOTE_WORK_FEATURE].mean(),
        }
    )

retention = pd.DataFrame(retention)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.lineplot(data=retention, x="threshold", y="retained_user_month_share", marker="o", ax=axes[0])
sns.lineplot(data=retention, x="threshold", y="retained_unique_users", marker="o", ax=axes[1])
sns.lineplot(data=retention, x="threshold", y="mean_wfh", marker="o", ax=axes[2])

axes[0].set_ylabel("Share of user-months retained")
axes[1].set_ylabel("Unique users retained")
axes[2].set_ylabel(f"Mean {REMOTE_WORK_FEATURE}")
for ax in axes:
    ax.axvline(MIN_TOTAL_WORK_DAYS, color="red", linestyle="--")
    ax.set_xlabel("Minimum total_work_days")

fig.suptitle("Sample-size and WFH stability tradeoff by threshold", y=1.05)
plt.tight_layout()
plt.show()

## Monthly retention at the chosen threshold

The chosen threshold should not create month-specific sample collapse.

In [ ]:
monthly_retention = (
    df_shares.assign(kept=df_shares["total_work_days"] >= MIN_TOTAL_WORK_DAYS)
    .groupby("month")
    .agg(
        total_user_months=("identifier", "size"),
        retained_user_months=("kept", "sum"),
        retained_unique_users=("identifier", lambda s: df_shares.loc[s.index[df_shares.loc[s.index, "total_work_days"] >= MIN_TOTAL_WORK_DAYS], "identifier"].nunique()),
    )
    .reset_index()
)
monthly_retention["retained_share"] = (
    monthly_retention["retained_user_months"] / monthly_retention["total_user_months"]
)

fig, ax1 = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly_retention, x="month", y="retained_share", marker="o", ax=ax1)
ax1.set_ylim(0, 1)
ax1.set_ylabel("Share retained")
ax1.set_xlabel("Month")
ax1.tick_params(axis="x", rotation=90)
ax1.set_title(f"Monthly retained sample after total_work_days >= {MIN_TOTAL_WORK_DAYS}")
plt.tight_layout()
plt.show()

## WFH trend sensitivity

Compare the selected WFH metric under nearby thresholds. The `>= 7` line should preserve the broad trend while reducing noise from weak user-months.

In [ ]:
thresholds_to_compare = [3, 5, 7, 10]
trend_rows = []

for threshold in thresholds_to_compare:
    trend = (
        df_shares[df_shares["total_work_days"] >= threshold]
        .groupby("month")[REMOTE_WORK_FEATURE]
        .mean()
        .reset_index()
    )
    trend["threshold"] = f">= {threshold}"
    trend_rows.append(trend)

trend_df = pd.concat(trend_rows, ignore_index=True)

plt.figure(figsize=(13, 5))
sns.lineplot(data=trend_df, x="month", y=REMOTE_WORK_FEATURE, hue="threshold", marker="o")
plt.title("Monthly WFH trend sensitivity to total_work_days threshold")
plt.xlabel("Month")
plt.ylabel("Mean WFH")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()